# 03. Inference: прогноз MAGT 2014-2024 + неопределённость + ALT

**Цель:** применить обученную ConvLSTM ко всем доступным годам, получить:
- Карты TTOP-прогноза (raw, без поправки)
- Карты MAGT (TTOP + latent heat correction)
- Оценку эпистемической неопределённости через MC Dropout
- Карту ALT (Active Layer Thickness) по формуле Стефана

**Входы:**
- `models/convlstm_ttop_rk_v2.pt` — обученная модель
- `data/tensor_01deg_v2.npz` — тензор признаков
- `data/landcover_v3.npz` (опционально) — landcover карта; если нет, считаем здесь

**Выходы:**
- `results/maps/predictions_2014_2024.npz` — все ежегодные карты
- `results/maps/uncertainty_mc_dropout.npz` — MC Dropout σ
- `results/maps/ALT_2023.npz` — ALT карта для 2023

In [ ]:
# Environment auto-detection
import os
from pathlib import Path

IN_COLAB_VM = (
    'COLAB_RELEASE_TAG' in os.environ or
    'COLAB_GPU' in os.environ
)

if IN_COLAB_VM:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    BASE_DIR = Path('/content/drive/MyDrive/AI4Arctic')
    env_label = 'Colab VM'
else:
    BASE_DIR = Path(os.environ.get(
        'AI4ARCTIC_HOME',
        Path.home() / 'Ai4Arctic'
    ))
    env_label = 'Local runtime'

print(f"Environment: {env_label}")
print(f"BASE_DIR: {BASE_DIR}")
assert BASE_DIR.exists(), f"BASE_DIR не найден: {BASE_DIR}"

import sys
sys.path.insert(0, str(BASE_DIR))

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(f"Device: {device}")

DATA_DIR = BASE_DIR / 'data'
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'
FIGURES_DIR = RESULTS_DIR / 'figures'
METRICS_DIR = RESULTS_DIR / 'metrics'
MAPS_DIR = RESULTS_DIR / 'maps'

for d in [MODELS_DIR, FIGURES_DIR, METRICS_DIR, MAPS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

## 1. Загружаем модель и данные

In [ ]:
from src.model import build_model_from_checkpoint
from src.data import normalize_features
from src.inference import predict_full_map, mc_dropout_predict, apply_latent_heat_correction
from src.landcover import (
    classify_landcover, build_delta_t_map, build_edaphic_map,
    stefan_alt, analyze_alt_by_landcover, landcover_stats,
)

# Модель
CKPT = MODELS_DIR / 'convlstm_ttop_rk_v2.pt'
model, y_mean, y_std, x_mean, x_std = build_model_from_checkpoint(CKPT, device=device)
print(f"Loaded model. y_mean={y_mean:.4f}, y_std={y_std:.4f}")

# Тензор
data = np.load(DATA_DIR / 'tensor_01deg_v2.npz')
X_full = data['X'].astype(np.float32)
lats, lons = data['lats'], data['lons']
print(f"X: {X_full.shape}, lats: {lats.shape}, lons: {lons.shape}")

# Применяем те же нормализационные статы, что использовались при обучении
X_norm = np.zeros_like(X_full)
for c in range(20):
    X_norm[..., c] = (X_full[..., c] - x_mean[c]) / x_std[c]
X_norm = np.nan_to_num(X_norm, nan=0.0).astype(np.float32)
print(f"X_norm: {X_norm.shape} готов для инференса")

## 2. Прогнозируем все доступные годы

Прогноз для года t использует признаки [t-4, t-1] как input.

In [ ]:
# Загружаем target для сравнения с прогнозами (если есть)
try:
    y_target = np.load(DATA_DIR / 'y_new_rk_landcover.npz')['y_new']
    has_target = True
except (FileNotFoundError, KeyError):
    y_target = None
    has_target = False

YEARS = list(range(2014, 2025))   # 2014..2024 (2024 без target)
YEAR_BASE = 2010                  # год t=0 в тензоре

predictions = {}
predictions_lh = {}  # с latent heat correction (заполним ниже)

# Сначала собираем все raw-предсказания
for year in YEARS:
    t_target = year - YEAR_BASE
    if t_target >= X_norm.shape[0]:
        # год за пределами доступных данных (2024 — t=14 если у тебя 14 годов)
        print(f"  {year}: t={t_target} вне X_norm (T={X_norm.shape[0]}), пропускаем")
        continue
    pred = predict_full_map(model, X_norm, t_target, y_mean, y_std, device=device)
    # Маскируем NaN там, где не было target
    if has_target and t_target < y_target.shape[0]:
        real = y_target[t_target]
        pred = np.where(np.isnan(real), np.nan, pred)
    predictions[year] = pred
    print(f"  {year}: pred mean = {np.nanmean(pred):+.2f}°C")

## 3. Landcover + latent heat correction

In [ ]:
# Берём landcover из data/ если есть, иначе считаем
LC_PATH = DATA_DIR / 'landcover_v3.npz'

if LC_PATH.exists():
    lc_data = np.load(LC_PATH)
    landcover = lc_data['landcover']
    print(f"Landcover загружен из {LC_PATH}")
else:
    print(f"Landcover не найден, считаем от scratch")
    # Используем средние NDVI/NDWI/soil_oc за все доступные годы
    # ВАЖНО: индексы каналов — должны совпасть с порядком в твоём тензоре
    X_mean = np.nanmean(X_full, axis=0)
    ndvi_mean = X_mean[..., 0]   # подкорректируй индекс под свой тензор
    ndwi_mean = X_mean[..., 1]
    soc_mean  = X_mean[..., 13]
    landcover = classify_landcover(ndvi_mean, ndwi_mean, soc_mean)
    np.savez_compressed(LC_PATH, landcover=landcover)

# Распределение
for cls, info in landcover_stats(landcover).items():
    print(f"  {cls} ({info['name']:>13}): {info['count']:>8,} ({info['fraction']:>5.1f}%)")

# Карта поправок
delta_t = build_delta_t_map(landcover)
print(f"\nMean delta_t: +{np.nanmean(delta_t):.2f}°C")

In [ ]:
# Применяем latent heat correction ко всем прогнозам
for year, pred in predictions.items():
    predictions_lh[year] = apply_latent_heat_correction(pred, delta_t)

# Сохраняем все
np.savez_compressed(
    MAPS_DIR / 'predictions_2014_2024.npz',
    **{f'pred_{y}': p for y, p in predictions.items()},
    **{f'pred_{y}_lh': p for y, p in predictions_lh.items()},
    lats=lats, lons=lons,
)
print(f"Сохранено: {MAPS_DIR / 'predictions_2014_2024.npz'}")

## 4. MC Dropout uncertainty (только для последнего года, 2023)

30 прогонов с активированным dropout — даёт эпистемическую неопределённость σ.

In [ ]:
TARGET_YEAR = 2023
t_target_idx = TARGET_YEAR - YEAR_BASE

if has_target and t_target_idx < y_target.shape[0]:
    nan_mask = np.isnan(y_target[t_target_idx])
else:
    nan_mask = None

mean_pred, std_pred = mc_dropout_predict(
    model, X_norm, t_target_idx, y_mean, y_std,
    n_samples=30, device=device, nan_mask=nan_mask, verbose=True
)

print(f"\nMC Dropout статистика σ (°C):")
print(f"  mean σ:   {np.nanmean(std_pred):.3f}")
print(f"  median σ: {np.nanmedian(std_pred):.3f}")
print(f"  P95 σ:    {np.nanpercentile(std_pred, 95):.3f}")

np.savez_compressed(
    MAPS_DIR / 'uncertainty_mc_dropout.npz',
    mean_pred=mean_pred, std_pred=std_pred,
    n_samples=30, year=TARGET_YEAR,
    lats=lats, lons=lons,
)
print(f"\nСохранено: {MAPS_DIR / 'uncertainty_mc_dropout.npz'}")

## 5. ALT (Active Layer Thickness) через формулу Стефана

In [ ]:
# TDD из тензора (8-й канал — Thawing Degree Days в К*сут)
# ВНИМАНИЕ: индекс канала — проверь под свой тензор
TDD_2023 = X_full[t_target_idx, ..., 8]
print(f"TDD 2023: mean {np.nanmean(TDD_2023):.0f}, max {np.nanmax(TDD_2023):.0f} K*day")

E_map = build_edaphic_map(landcover)

# Permafrost mask = pred_2023_lh < 0
pred_2023_lh = predictions_lh[TARGET_YEAR]
pf_mask = pred_2023_lh < 0

ALT_2023 = stefan_alt(TDD_2023, E_map, permafrost_mask=pf_mask)
print(f"ALT 2023: mean {np.nanmean(ALT_2023):.2f}m, "
      f"p95 {np.nanpercentile(ALT_2023, 95):.2f}m, "
      f"p5 {np.nanpercentile(ALT_2023, 5):.2f}m")

# Статистика по landcover
print("\nALT по landcover:")
for cls, info in analyze_alt_by_landcover(ALT_2023, landcover).items():
    print(f"  {cls} ({info['name']:>13}): mean {info['mean_m']:.2f}m, "
          f"p95 {info['p95_m']:.2f}m  (n={info['n']:,})")

np.savez_compressed(
    MAPS_DIR / 'ALT_2023.npz',
    ALT=ALT_2023, E_map=E_map, TDD=TDD_2023,
    lats=lats, lons=lons,
)
print(f"\nСохранено: {MAPS_DIR / 'ALT_2023.npz'}")

## 6. Карты для визуального осмотра

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# MAGT 2023
im0 = axes[0, 0].pcolormesh(lons, lats, pred_2023_lh, cmap='RdBu_r',
                              vmin=-12, vmax=2, shading='auto')
axes[0, 0].set_title('MAGT 2023 (ConvLSTM + lh correction)')
plt.colorbar(im0, ax=axes[0, 0], label='°C', fraction=0.03)

# Uncertainty σ
im1 = axes[0, 1].pcolormesh(lons, lats, std_pred, cmap='viridis',
                              vmin=0, vmax=1, shading='auto')
axes[0, 1].set_title(f'MC Dropout σ (n=30)')
plt.colorbar(im1, ax=axes[0, 1], label='°C', fraction=0.03)

# Landcover
im2 = axes[1, 0].pcolormesh(lons, lats, landcover, cmap='tab10',
                              vmin=0, vmax=5, shading='auto')
axes[1, 0].set_title('Landcover classes (0=undef, 1=peat, 2=taiga, 3=tundra, 4=bare)')
plt.colorbar(im2, ax=axes[1, 0], fraction=0.03)

# ALT
im3 = axes[1, 1].pcolormesh(lons, lats, ALT_2023, cmap='YlOrRd',
                              vmin=0, vmax=2.5, shading='auto')
axes[1, 1].set_title('ALT 2023 (m)')
plt.colorbar(im3, ax=axes[1, 1], label='m', fraction=0.03)

for ax in axes.flat:
    ax.set_xlabel('Долгота, °E'); ax.set_ylabel('Широта, °N')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'inference_summary_2023.png', dpi=150, bbox_inches='tight')
plt.show()